# Poker McPokerface — Human vs LLM Bots

An interactive notebook for playing 5-card draw against three local-LLM bots, each driven by a different personality. This is the assessor-friendly counterpart to `submission.ipynb`'s analytical pipeline — same engine and bots, but here you sit at the table.

**Setup time:** ~10 minutes on first run (Ollama install + 3 model pulls). Subsequent runs in the same session are instant.

**How to play:** at each prompt, type one of `fold`, `check`, `call`, or `raise N` (where `N` is the chip amount). On the draw round, type 0–5 comma-separated card indices to swap (e.g. `0,2,4`, `all` for all five, or empty to stand pat).


## 1. Setup

Detects the environment, clones the repo (Colab only), installs Ollama and pulls the three models we need. Skip this cell if you've already run `submission.ipynb` in the same Colab session — the daemon and models will still be live.


In [ ]:
# 1. One-shot setup. Idempotent — safe to run twice.
import os, sys, subprocess, time, shutil

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/adambrennan150/PokerAI.git'
REPO_DIR = 'PokerAI'

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print('Cloning repo...')
        subprocess.check_call(['git', 'clone', '-q', REPO_URL])
    if os.getcwd().split('/')[-1] != REPO_DIR:
        os.chdir(REPO_DIR)
    print('cwd:', os.getcwd())

    # Install Python deps
    subprocess.check_call(['pip', 'install', '-q', '-r', 'requirements.txt'])

    # Install Ollama (zstd first — Colab base image lacks it and the
    # current Ollama installer requires it for extraction).
    if shutil.which('ollama') is None:
        print('Installing Ollama...')
        subprocess.check_call(['apt-get', 'update', '-qq'])
        subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'zstd'])
        subprocess.check_call('curl -fsSL https://ollama.com/install.sh | sh', shell=True)

    # Start the daemon as a background process
    if not os.path.exists('/tmp/ollama.pid') or subprocess.run(['pgrep', '-f', 'ollama serve'], capture_output=True).returncode != 0:
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)
        print('Ollama daemon started.')
else:
    print('Local run — assuming repo, deps, and a running Ollama daemon are already set up.')

# Make repo modules importable
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Pull the three models we play against
MODELS = ['llama3.1:8b', 'mistral', 'gemma3:1b']
for m in MODELS:
    print(f'\n--- Pulling {m} ---')
    subprocess.run(['ollama', 'pull', m])

print('\nSetup complete.')


## 2. Play a hand

Builds a 4-seat table — you, plus three LLM bots with distinct personalities — and plays one full hand of 5-card draw. Each hand takes ~30–60 seconds depending on how long the bots think. Re-run the cell to play another hand from a fresh deck (chips reset to 200 each time).


In [ ]:
# 2. Single hand against three LLM bots.
from engine import Player, Seat, Game
from bots import OllamaBot
from bots.personalities import TIGHT_AGGRESSIVE, LOOSE_AGGRESSIVE, BLUFFER
from config.models import BY_ID
from ui import HumanAgent


def _llm(name, personality, model_id):
    spec = BY_ID[model_id]
    return OllamaBot(
        name=name, personality=personality, model_id=model_id,
        num_predict=spec.num_predict,
        system_prefix=spec.system_prefix,
        think=spec.think,
    )


starting_chips = 200
seats = [
    Seat(player=Player(name='You',          chips=starting_chips), agent=HumanAgent(name='You')),
    Seat(player=Player(name='TAG-Llama',    chips=starting_chips), agent=_llm('TAG-Llama',    TIGHT_AGGRESSIVE, 'llama3.1:8b')),
    Seat(player=Player(name='LAG-Mistral',  chips=starting_chips), agent=_llm('LAG-Mistral',  LOOSE_AGGRESSIVE, 'mistral')),
    Seat(player=Player(name='Bluff-Gemma',  chips=starting_chips), agent=_llm('Bluff-Gemma',  BLUFFER,          'gemma3:1b')),
]

summary = Game(seats=seats, ante=1, min_bet=5, seed=None).play_hand()

print('\n=== hand summary ===')
for r in summary.seat_results:
    flag = '  (folded)' if r.folded else ''
    print(f'  {r.name:14s}  start: {r.starting_chips:4d}  end: {r.ending_chips:4d}  net: {r.net_change:+5d}{flag}')
if summary.winners:
    print('\nwinners:', ', '.join(f'{n} (+{c})' for n, c in summary.winners))


---

Want to look at *why* the bots played the way they did? Their reasoning is captured in `bot.last_response.reasoning_text` — re-run the cell above and afterwards inspect any of `seats[1].agent`, `seats[2].agent`, `seats[3].agent` to see the parsed reasoning from their most recent decision.
